# 02 — Final seven-category classifier

Run top to bottom: load prepared data, explore it, split by normalized description, compare models, tune, calibrate, evaluate, and save. Training takes several minutes. This workflow never loads an older model over a newly trained one.

Data is synthetic and labels are provisional. Group separation limits repeated-text leakage but does not hold out all merchants. Previous test results were explored during development; this is not an untouched external benchmark.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Find the project whether this notebook runs from root or notebooks/.
PROJECT_PATH = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "requirements.txt").exists() and (p / "src").is_dir()
)
if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))

data_path = PROJECT_PATH / "data/raw/processed/expenses_labeled_v1.csv"
expense_df = pd.read_csv(data_path)
print("Loaded rows:", len(expense_df))


In [ ]:
submission_df = expense_df.copy()

extra_categories = [
    "Housing",
    "Insurance",
    "Personal Care",
    "Education",
    "Fees"
]

submission_df["target_category"] = (
    submission_df["target_category"].replace(
        extra_categories, "Other"
    )
)

print(submission_df["target_category"].value_counts())
print("\nMissing labels:", submission_df["target_category"].isna().sum())
print("Categories:", submission_df["target_category"].nunique())

In [ ]:
import matplotlib.pyplot as plt

category_counts = submission_df["target_category"].value_counts()

category_counts.plot(kind="bar", figsize=(9, 4))
plt.title("Expenses per category")
plt.xlabel("Category")
plt.ylabel("Number of expenses")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

text_lengths = submission_df["description"].str.len()

text_lengths.plot(kind="hist", bins=30, figsize=(9, 4))
plt.title("Transaction description lengths")
plt.xlabel("Length in characters")
plt.ylabel("Number of descriptions")
plt.tight_layout()
plt.show()

print("Description length summary:")
print(text_lengths.describe())

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

for category in sorted(submission_df["target_category"].unique()):
    descriptions = submission_df.loc[
        submission_df["target_category"] == category,
        "description"
    ]

    vectorizer = CountVectorizer(
        stop_words="english",
        token_pattern=r"\b[a-zA-Z]{2,}\b"
    )

    word_counts = vectorizer.fit_transform(descriptions)

    counts = pd.Series(
        word_counts.sum(axis=0).A1,
        index=vectorizer.get_feature_names_out()
    )

    print(f"\n--- {category} ---")
    print(counts.nlargest(10))

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from src.text_processing import clean_text

banking_words = {
    "debit", "credit", "payment", "withdrawal",
    "paypal", "pypl", "pp", "inst", "xfer",
    "web", "id", "ppd", "usa"
}

stop_words = sorted(set(ENGLISH_STOP_WORDS) | banking_words)

print(clean_text("[debit] UBER*123 Ride to Airport!"))

In [ ]:
normalized = submission_df["description"].map(clean_text)

label_counts = submission_df.groupby(normalized)[
    "target_category"
].nunique()

conflict_keys = label_counts[label_counts > 1].index

has_conflict = normalized.isin(conflict_keys)

review_df = submission_df.loc[has_conflict].copy()
model_df = submission_df.loc[~has_conflict].copy()

groups = normalized.loc[model_df.index]

print("Rows set aside:", len(review_df))
print("Modeling rows:", len(model_df))

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

first_split = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)
#creating the arrays 
remaining_pos, test_pos = next(
    first_split.split(model_df, groups=groups)
)

remaining_df = model_df.iloc[remaining_pos].copy()
test_df = model_df.iloc[test_pos].copy()

remaining_groups = groups.iloc[remaining_pos]

second_split = GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=43
)

train_pos, val_pos = next(
    second_split.split(remaining_df, groups=remaining_groups)
)

train_df = remaining_df.iloc[train_pos].copy()
val_df = remaining_df.iloc[val_pos].copy()

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
train_groups = set(normalized.loc[train_df.index])
val_groups = set(normalized.loc[val_df.index])
test_groups = set(normalized.loc[test_df.index])

print("Train/validation overlap:", len(train_groups & val_groups))
print("Train/test overlap:", len(train_groups & test_groups))
print("Validation/test overlap:", len(val_groups & test_groups))

print(pd.DataFrame({
    "train": train_df["target_category"].value_counts(),
    "validation": val_df["target_category"].value_counts(),
    "test": test_df["target_category"].value_counts()
}).fillna(0).astype(int))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

classifiers = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=42
    ),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(random_state=42)
}

trained_models = {}
results = []

for name, classifier in classifiers.items():
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            preprocessor=clean_text,
            stop_words=stop_words,
            ngram_range=(1, 2),
            min_df=2
        )),
        ("classifier", classifier)
    ])

    pipeline.fit(
        train_df["description"],
        train_df["target_category"]
    )

    predictions = pipeline.predict(val_df["description"])

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(
            val_df["target_category"], predictions
        ),
        "Macro F1": f1_score(
            val_df["target_category"],
            predictions,
            average="macro",
            zero_division=0
        )
    })

    trained_models[name] = pipeline

comparison_df = pd.DataFrame(results).sort_values(
    "Macro F1", ascending=False
)

print(comparison_df.to_string(index=False))

In [ ]:
from sklearn.model_selection import GridSearchCV, GroupKFold

parameter_grid = {
    "tfidf__min_df": [1, 2],
    "classifier__C": [0.5, 1.0, 2.0]
}

search = GridSearchCV(
    estimator=trained_models["Linear SVM"],
    param_grid=parameter_grid,
    scoring="f1_macro",
    cv=GroupKFold(n_splits=3),
    n_jobs=1
)

search.fit(
    train_df["description"],
    train_df["target_category"],
    groups=normalized.loc[train_df.index]
)

print("Best settings:", search.best_params_)
print("Cross-validation macro F1:", search.best_score_)

In [ ]:
tuned_model = search.best_estimator_

val_predictions = tuned_model.predict(val_df["description"])

print("Validation accuracy:", accuracy_score(
    val_df["target_category"], val_predictions
))

print("Validation macro F1:", f1_score(
    val_df["target_category"],
    val_predictions,
    average="macro",
    zero_division=0
))

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

calibration_folds = list(
    GroupKFold(n_splits=3).split(
        train_df["description"],
        train_df["target_category"],
        groups=normalized.loc[train_df.index]
    )
)

final_model = CalibratedClassifierCV(
    estimator=tuned_model,
    method="sigmoid",
    cv=calibration_folds
)

final_model.fit(
    train_df["description"],
    train_df["target_category"]
)

calibrated_predictions = final_model.predict(
    val_df["description"]
)

print("Calibrated validation accuracy:", accuracy_score(
    val_df["target_category"], calibrated_predictions
))

print("Calibrated validation macro F1:", f1_score(
    val_df["target_category"],
    calibrated_predictions,
    average="macro",
    zero_division=0
))


In [ ]:
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay
)

test_predictions = final_model.predict(test_df["description"])

print("Test accuracy:", accuracy_score(
    test_df["target_category"], test_predictions
))

print(classification_report(
    test_df["target_category"],
    test_predictions,
    digits=3,
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    test_df["target_category"],
    test_predictions,
    labels=final_model.classes_,
    xticks_rotation=45,
    cmap="Blues",
    values_format="d"
)

plt.title("Final model — test confusion matrix")
plt.tight_layout()
report_path = PROJECT_PATH / "reports"
report_path.mkdir(exist_ok=True)
plt.savefig(report_path / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import joblib

# Cleaning was imported from src before training, so it is portable.
model_path = PROJECT_PATH / "models/expense_classifier_final.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, model_path)
loaded_model = joblib.load(model_path)
print("Saved and loaded:", model_path)
print(loaded_model.predict(["Netflix monthly payment"]))


In [ ]:
import src.predict as prediction_api

# Refresh the cached model if this notebook saved a newer model above.
prediction_api._model = None
print(prediction_api.predict("Netflix monthly payment"))
